# Advanced Agentic Patterns — A Practical Guide
## LangGraph 1.x + Ollama (qwen3.5) — What, Why, How & When


---

### What this notebook covers (8 patterns)

| # | Pattern | One-line idea | Real-world use |
|---|---------|---------------|----------------|
| 1 | **ReAct Agent** | Reason → Act → Observe, in a loop | The default tool-using agent |
| 2 | **Structured Output** | Force the LLM to return clean JSON / objects | APIs, databases, pipelines |
| 3 | **Reflection** | Agent critiques and improves its own answer | Writing, code, reasoning quality |
| 4 | **Plan-and-Execute** | First make a plan, then execute each step | Complex multi-step tasks |
| 5 | **Human-in-the-Loop** | Pause and ask a human before risky actions | Approvals, safety, compliance |
| 6 | **Multi-Agent Supervisor** | A coordinator agent routes work to specialists | Enterprise systems, division of labour |
| 7 | **Long-Term Memory** | Remember facts *across* sessions | Personal assistants, CRM |
| 8 | **Agentic RAG** | Agent *decides* when to retrieve & self-checks | Smart Q&A over documents |

### How each pattern is laid out

Every pattern follows the same 4 parts, so you can read them in order or jump to the one you need:

1. **🧠 Theory** — *what* it is and *why* it matters
2. **📐 How it works** — the diagram / mental model
3. **💻 Code** — a small, runnable example
4. **🎯 Scenario** — *when* to use it, plus a hands-on exercise to try

---
**Stack:** `langgraph` 1.x · `langchain-ollama` · `qwen3.5:4b` (switch to `qwen3.5:9b` for stronger reasoning)


---
## Setup

Run these three cells once. Everything below reuses the `llm` object they create.

In [ ]:
# ── Install / verify packages ─────────────────────────────────────────────────
import subprocess, sys

packages = [
    "langgraph",          # the agent orchestration framework
    "langchain-ollama",   # Ollama LLM integration
    "langchain-core",     # tools, messages, prompts
    "pydantic",           # structured output schemas
    "ollama",             # direct Ollama client (health checks)
]
for pkg in packages:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(("✓ " if r.returncode == 0 else "⚠ ") + pkg)

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import warnings, json, operator
from uuid import uuid4
from typing import TypedDict, Annotated, Literal
warnings.filterwarnings("ignore")

import ollama
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage
)

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent, ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import interrupt, Command

print("✓ All imports successful")

In [ ]:
# ── Configuration & Model Setup ──────────────────────────────────────────────

# qwen3.5 models support native tool/function calling — essential for agents
MODEL = "qwen3.5:4b"   # switch to "qwen3.5:9b" for stronger reasoning / planning

print(f"Testing model: {MODEL}")
try:
    available = [m.model for m in ollama.list().models]
    print(f"Available: {available}")

    if MODEL not in available:
        for fallback in ["qwen3.5:9b", "qwen3.5:4b", "qwen2.5:0.5b"]:
            if fallback in available:
                MODEL = fallback
                print(f"  Using fallback: {MODEL}")
                break

    resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": "Say: Agent ready"}],
        options={"num_predict": 10},
    )
    print(f"  ✅ {MODEL} → {resp.message.content.strip()}")
except Exception as e:
    print(f"  ❌ {e}")
    print("     Ensure Ollama is running: ollama serve")

# The shared LLM object — every pattern below reuses this
llm = ChatOllama(model=MODEL, temperature=0, num_ctx=8192)
print(f"\n  Active model: {MODEL}")

### A few shared tools

Several patterns below need tools to call. We define two tiny, safe ones here and reuse them.

In [ ]:
# ── Shared tools (reused across patterns) ─────────────────────────────────────

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '144 / 12' or '23 * 7 + 4'.
    Use this for ANY math — never compute in your head.
    """
    import ast, operator as op
    ops = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
           ast.Div: op.truediv, ast.Pow: op.pow, ast.Mod: op.mod,
           ast.USub: op.neg}
    def ev(node):
        if isinstance(node, ast.Constant):          return node.value
        if isinstance(node, ast.BinOp):             return ops[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):           return ops[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    try:
        return str(ev(ast.parse(expression, mode="eval").body))
    except Exception as e:
        return f"Error: {e}"


@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())


print("Tools ready:", calculator.name, "|", word_count.name)
print("Test:", calculator.invoke({"expression": "144 / 12 + 1"}))

---
# Pattern 1 — ReAct Agent (Reason + Act)

## 🧠 Theory — What & Why

**ReAct = Reasoning + Acting.** It is the *foundational* pattern for every tool-using agent.

The idea (from the 2022 ReAct paper) is simple but powerful: instead of the LLM answering
in one shot, it **interleaves thinking and doing** in a loop:

```
Thought  → "I need to calculate 144 / 12"
Action   → calls calculator("144 / 12")
Observation → "12"
Thought  → "Now I have the answer"
Answer   → "The result is 12"
```

### Why it matters
- A plain LLM **hallucinates** math, dates, and facts. ReAct lets it *use a tool* instead of guessing.
- The loop **self-corrects**: if a tool errors, the agent sees the error and tries again.
- It is the **building block** — patterns 3–8 are all ReAct loops with extra structure around them.

### Why we don't hand-build it here
In the previous notebook you built this loop *by hand* with `StateGraph`, `ToolNode`, and
`tools_condition`. That was great for *understanding*. In production you use the
**`create_react_agent`** prebuilt — one line, battle-tested, with streaming + memory built in.


## 📐 How it works

```
                    ┌─────────────────────────────────┐
   user message ──▶ │            create_react_agent   │
                    │                                 │
                    │   ┌─────────┐   tool_calls?     │
                    │   │   LLM   │──── yes ──▶ run tool│
                    │   │ (reason)│◀─── observation ───┘│
                    │   └────┬────┘                     │
                    │        │ no more tool_calls       │
                    └────────┼─────────────────────────┘
                             ▼
                       final answer
```

One function replaces ~20 lines of graph wiring.

In [ ]:
# ── Pattern 1: ReAct agent in ONE line ────────────────────────────────────────

react_agent = create_react_agent(
    model=llm,
    tools=[calculator, word_count],
    prompt="You are a helpful assistant. Use the calculator for any math "
           "and word_count for counting words. Always use a tool when relevant.",
)

def run_react(question: str):
    result = react_agent.invoke({"messages": [HumanMessage(question)]})
    print(f"❓ {question}\n")
    for m in result["messages"]:
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"   🔧 Action: {tc['name']}({tc['args']})")
        elif isinstance(m, ToolMessage):
            print(f"   👁  Observation: {m.content}")
    print(f"\n🤖 Answer: {result['messages'][-1].content}")

run_react("What is 144 divided by 12, and how many words are in "
          "the phrase 'machine learning is fun'?")

## 🎯 Scenario — When to use ReAct

> **Use ReAct whenever the answer needs an *action* — a calculation, a lookup, an API call, a database query.**

**Real-world example:** *"You're building a customer-support bot. The user asks 'What's my order total
including 8% tax on $245?' A plain LLM might get the math wrong. A ReAct agent calls a `calculator`
tool and gets it exactly right, every time."*

**🧪 Try it yourself:** Add a third tool `reverse_string(text)` and ask the agent
*"Reverse the word 'agent' and tell me how many letters it has."* Watch it chain two tools.

---
# Pattern 2 — Structured Output

## 🧠 Theory — What & Why

By default an LLM returns **free-form text**. But software needs **structured data** —
a Python object or JSON with known fields. Structured output **forces** the model to fill
in a schema you define.

### Why it matters
- **Downstream code** needs reliable fields: `card.difficulty`, not "somewhere in this paragraph".
- **No parsing nightmares** — you get a validated Pydantic object, not a regex hunt.
- **APIs & databases** — you can directly store the result or return it as JSON.

### ⚠️ Critical lesson for local models
LangGraph's `create_react_agent(response_format=...)` uses a *JSON-parsing* strategy by default,
which **frequently fails on small local models** (qwen3.5:4b returns prose, the parser crashes).

✅ **The reliable path is `llm.with_structured_output(Schema)`** — it uses Ollama's **native
JSON-schema mode**, where the model is *constrained* to produce valid JSON. This works far better
on small models. *(This is the kind of detail that separates a demo from a production system.)*


In [ ]:
# ── Pattern 2: Structured output with Pydantic ────────────────────────────────

class ConceptCard(BaseModel):
    """A study flashcard for an ML concept."""
    term: str = Field(description="the ML term being explained")
    definition: str = Field(description="a clear one-sentence definition")
    analogy: str = Field(description="a simple real-world analogy")
    difficulty: Literal["beginner", "intermediate", "advanced"] = Field(
        description="how hard the concept is")

# with_structured_output → native JSON-schema mode (reliable on local models)
card_maker = llm.with_structured_output(ConceptCard)

card = card_maker.invoke("Explain the concept of 'overfitting' as a study flashcard.")

print("Returned a Python object:", type(card).__name__)
print(f"  term       : {card.term}")
print(f"  definition : {card.definition}")
print(f"  analogy    : {card.analogy}")
print(f"  difficulty : {card.difficulty}")
print()
print("As JSON for an API:", card.model_dump_json(indent=2))

## 🎯 Scenario — When to use Structured Output

> **Use it whenever the LLM output feeds into *code* instead of a *human* eye.**

**Real-world example:** *"You're extracting data from 10,000 resumes. You don't want a paragraph —
you want `{name, years_experience, skills[], seniority}` for every single one, so you can load it
into a database. Structured output guarantees every result has exactly those fields."*

**🧪 Try it yourself:** Define a `SentimentResult` schema with fields `label`
(`Literal["positive","negative","neutral"]`), `confidence` (float 0–1), and `key_phrase` (str).
Run it on the review *"The course was okay but the pace was too fast."*

---
# Pattern 3 — Reflection (Self-Critique)

## 🧠 Theory — What & Why

A first draft is rarely the best draft — **for humans and for LLMs**. The Reflection pattern
adds a **critic** that reviews the output and sends feedback back to the generator, which then
**revises**. Loop until the critic is satisfied (or you hit a max).

```
        ┌──────────────┐         ┌──────────────┐
        │  GENERATE    │────────▶│   REFLECT    │
        │ (write draft)│         │ (critique it)│
        └──────▲───────┘         └──────┬───────┘
               │   "improve this:"      │
               └────────────────────────┘
                  loop until APPROVED
```

### Why it matters
- **Quality jumps** dramatically — the second draft fixes what the first missed.
- It mimics how **experts actually work**: write → review → rewrite.
- Used for: essay writing, **code generation** (generate → test → fix), complex reasoning.

### The trick
The *same* LLM plays **two roles**: first the **author**, then a **strict reviewer**.
Different prompts give it different "hats."


In [ ]:
# ── Pattern 3: Reflection loop (generate → reflect → revise) ───────────────────

class ReflectState(TypedDict):
    task: str          # what to explain
    draft: str         # current draft
    critique: str      # reviewer feedback
    iteration: int     # safety counter

def generate(state: ReflectState):
    """Author hat: write (or rewrite) the explanation."""
    if state.get("critique"):
        prompt = (f"Rewrite and improve your explanation of '{state['task']}'.\n\n"
                  f"Previous draft:\n{state['draft']}\n\n"
                  f"Reviewer feedback to address:\n{state['critique']}")
    else:
        prompt = f"Write a clear 3-4 sentence explanation of: {state['task']}"
    draft = llm.invoke(prompt).content
    return {"draft": draft, "iteration": state.get("iteration", 0) + 1}

def reflect(state: ReflectState):
    """Reviewer hat: critique strictly, or APPROVE."""
    prompt = (f"You are a strict ML professor reviewing an explanation of "
              f"'{state['task']}'. If it is accurate, clear and complete, reply with "
              f"exactly the single word: APPROVED. Otherwise, give 1-2 specific, "
              f"actionable improvements.\n\nExplanation:\n{state['draft']}")
    return {"critique": llm.invoke(prompt).content}

def keep_going(state: ReflectState):
    """Stop if approved OR after 3 rounds (don't loop forever)."""
    if "APPROVED" in state["critique"].upper() or state["iteration"] >= 3:
        return END
    return "generate"

rg = StateGraph(ReflectState)
rg.add_node("generate", generate)
rg.add_node("reflect", reflect)
rg.add_edge(START, "generate")
rg.add_edge("generate", "reflect")
rg.add_conditional_edges("reflect", keep_going, {END: END, "generate": "generate"})
reflection_app = rg.compile()

result = reflection_app.invoke({"task": "why we use the ReLU activation function"})
print(f"Finished after {result['iteration']} draft(s)\n")
print("FINAL DRAFT:\n", result["draft"])
print("\nLAST CRITIQUE:\n", result["critique"][:300])

In [ ]:
# # See the structure:
# print(reflection_app.get_graph().draw_ascii())

# # Firehose: LangGraph prints every node entry/exit + full state:
# reflection_app.invoke({"task":"why we use the ReLU activation function"}, debug=True)


In [ ]:
reflection_app

## 🎯 Scenario — When to use Reflection

> **Use it when *quality* matters more than *speed* — and when "good enough on the first try" isn't.**

**Real-world example:** *"An AI writes marketing copy. The first version is generic. A reflection
loop adds a 'brand reviewer' that says 'too salesy, make it more concrete' — and the rewrite is
noticeably better. The same idea powers AI coding tools: generate code → run tests → read the
error → fix → repeat."*

**🧪 Try it yourself:** Change the reviewer to a "5-year-old test" — it only APPROVES if the
explanation would make sense to a child. Watch how the drafts get simpler each round.

---
# Pattern 4 — Plan-and-Execute

## 🧠 Theory — What & Why

A ReAct agent decides its **next** step one at a time. For **complex multi-step tasks**, that can
drift or lose the thread. Plan-and-Execute fixes this by **separating thinking from doing**:

1. A **Planner** looks at the whole objective and writes an explicit **ordered plan**.
2. An **Executor** carries out each step in order.

```
objective ─▶ ┌─────────┐  plan = [step1, step2, step3] ─▶ ┌──────────┐
             │ PLANNER │                                   │ EXECUTOR │
             └─────────┘                                   │ do each  │
                                                           └────┬─────┘
                                                                ▼
                                                          final result
```

### Why it matters
- **Stays on track** for long tasks — the plan is a checklist it won't forget.
- **Transparent & debuggable** — you can *see* the plan before anything runs.
- **Cheaper reasoning** — the expensive "think hard" step happens once, up front.

> 💡 The *advanced* version also **re-plans**: after executing, it asks "are we done, or do we
> need to adjust the plan?" We keep it linear here for clarity and note re-planning as an extension.


In [ ]:
# ── Pattern 4: Plan-and-Execute ───────────────────────────────────────────────

class Plan(BaseModel):
    """An ordered list of concrete steps to accomplish an objective."""
    steps: list[str] = Field(description="exactly 3 concrete, ordered steps")

class PlanState(TypedDict):
    objective: str
    plan: list
    result: str

planner_llm = llm.with_structured_output(Plan)            # reliable structured plan
# qwen3.5 is a *reasoning* model — reasoning=False skips the slow "thinking"
# phase, so each execution step is fast and returns clean, non-empty content.
llm_call = ChatOllama(model=MODEL, temperature=0, reasoning=False)

def plan_node(state: PlanState):
    """Look at the whole objective and produce an ordered 3-step plan."""
    p = planner_llm.invoke(
        f"Create a 3-step plan to accomplish this objective. "
        f"Each step is one short actionable instruction.\n\n"
        f"Objective: {state['objective']}")
    return {"plan": p.steps[:3]}                            

def execute_node(state: PlanState):
    """Execute each step of the plan in order (briefly)."""
    done = []
    for i, step in enumerate(state["plan"], 1):
        out = llm_call.invoke(
            f"Objective: {state['objective']}\n"
            f"Do ONLY this step in 1-2 sentences:\nStep {i}: {step}").content
        done.append(f"Step {i}: {step}\n   → {out.strip()[:250]}")
    return {"result": "\n\n".join(done)}

pg = StateGraph(PlanState)
pg.add_node("plan", plan_node)
pg.add_node("execute", execute_node)
pg.add_edge(START, "plan")
pg.add_edge("plan", "execute")
pg.add_edge("execute", END)
planexec_app = pg.compile()

out = planexec_app.invoke({
    "objective": "Design a beginner mini-project to teach a beginner about decision trees"})
print("📋 THE PLAN:")
for i, s in enumerate(out["plan"], 1):
    print(f"  {i}. {s}")
print("\n" + "=" * 60)
print("⚙️  EXECUTION:\n")
print(out["result"])

## 🎯 Scenario — When to use Plan-and-Execute

> **Use it for tasks with several dependent steps where order matters and the agent could otherwise wander.**

**Real-world example:** *"A research assistant agent gets 'Write me a report comparing 3 ML
frameworks.' Without a plan it rambles. With Plan-and-Execute it first writes the plan —
(1) list the frameworks, (2) compare on speed, (3) compare on ease-of-use, (4) write a verdict —
then executes each. The output is organised and complete."*

**🧪 Try it yourself:** Add a `print` inside `execute_node` so you can watch each step run live.
Then give it the objective *"Plan a 5-day study schedule for the ML exam."*

---
# Pattern 5 — Human-in-the-Loop (HITL)

## 🧠 Theory — What & Why

Fully autonomous agents are great — until one decides to **delete a database** or **send 10,000
emails**. Human-in-the-Loop inserts an **approval gate**: the agent **pauses**, shows a human what
it's about to do, and **waits** for a yes/no before continuing.

```
   ... ─▶ [prepare action] ─▶ ⏸  INTERRUPT  ⏸ ─▶ [act] ─▶ ...
                                    │
                            human reviews & approves
```

### Why it matters
- **Safety** — a human gates irreversible or expensive actions.
- **Compliance** — finance, healthcare, legal *require* human sign-off.
- **Trust** — users adopt agents faster when they stay in control.

### How LangGraph does it
- `interrupt(payload)` **pauses** the graph and surfaces `payload` to your app.
- The state is **saved by the checkpointer** (so you need one).
- You **resume** by calling the graph again with `Command(resume=<human_decision>)`.
- Execution continues **exactly where it left off** — the human's answer becomes `interrupt`'s return value.


In [ ]:
# ── Pattern 5: Human-in-the-loop approval gate ────────────────────────────────

class HITLState(TypedDict):
    request: str
    approved: str
    result: str

def approval_gate(state: HITLState):
    """Pause and ask a human to approve the action."""
    decision = interrupt({                       # ⏸ graph pauses HERE
        "action_to_approve": state["request"],
        "instructions": "Reply 'yes' to execute or 'no' to cancel.",
    })
    return {"approved": decision}                # resumes with the human's reply

def act(state: HITLState):
    if str(state["approved"]).strip().lower().startswith("y"):
        return {"result": f"✅ EXECUTED: {state['request']}"}
    return {"result": f"❌ CANCELLED by human: {state['request']}"}

hg = StateGraph(HITLState)
hg.add_node("approval_gate", approval_gate)
hg.add_node("act", act)
hg.add_edge(START, "approval_gate")
hg.add_edge("approval_gate", "act")
hg.add_edge("act", END)

# HITL REQUIRES a checkpointer (state must survive the pause)
hitl_app = hg.compile(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "approval-demo-1"}}

# ── Turn 1: the agent runs until it hits the approval gate, then PAUSES ───────
paused = hitl_app.invoke(
    {"request": "DELETE all user records for batch 'Jan26'"}, cfg)
print("⏸  GRAPH PAUSED — waiting for human approval")
print("   Agent wants to:", paused["__interrupt__"][0].value["action_to_approve"])

In [ ]:
# ── Turn 2: a HUMAN decides. Resume with Command(resume=...) ──────────────────


human_decision = "no"     

final = hitl_app.invoke(Command(resume=human_decision), cfg)
print(f"Human said: '{human_decision}'")
print("Result:", final["result"])

## 🎯 Scenario — When to use Human-in-the-Loop

> **Use it before any action that is irreversible, expensive, or high-risk.**

**Real-world example:** *"An agent manages a company's cloud bill. It finds an idle server and wants
to shut it down. HITL pauses: 'About to terminate server prod-db-07 — approve?' An engineer checks,
sees it's actually the production database, and clicks NO. The interrupt just prevented an outage."*

**🧪 Try it yourself:** Re-run the two cells with `human_decision = "yes"` and watch the action
execute. Then add a second sensitive action and gate both.

---
# Pattern 6 — Multi-Agent Supervisor

## 🧠 Theory — What & Why

One agent with 20 tools gets **confused**. The production answer is **specialisation**: build
several *focused* agents and put a **Supervisor** in charge of routing — exactly like a manager
delegating to a team.

```
                      ┌────────────────┐
        user ───────▶ │   SUPERVISOR   │ ◀───── (specialists report back)
                      │  "who should   │
                      │   handle this?"│
                      └───┬────────┬───┘
              ┌───────────┘        └───────────┐
              ▼                                 ▼
       ┌─────────────┐                   ┌─────────────┐
       │ math_expert │                   │  ml_tutor   │
       └─────────────┘                   └─────────────┘
```

### Why it matters
- **Better accuracy** — each specialist has a narrow, clear job and a focused prompt.
- **Scalable** — add a new skill = add a new specialist, no rewrite of the others.
- **The enterprise standard** — this supervisor-worker hierarchy is how real multi-agent
  systems (customer service, research, coding teams) are built.

### How it routes (the key trick)
The supervisor returns a **`Command(goto=...)`** — a dynamic jump to whichever node it picks.
We use **structured output** so the routing decision is a clean enum, not free text. Each
specialist does its job, then **reports back to the supervisor**, who decides `FINISH` or routes
again. A **step counter** guarantees the loop always terminates.


In [ ]:
# ── Pattern 6: Multi-agent supervisor ─────────────────────────────────────────

class SupervisorState(MessagesState):   
    steps: int                          

class Route(BaseModel):
    """Which specialist should act next (or FINISH if the user is fully answered)."""
    next: Literal["math_expert", "ml_tutor", "FINISH"]

def _render(messages):
    out = []
    for m in messages:
        role = type(m).__name__.replace("Message", "")
        name = getattr(m, "name", None)
        out.append(f"{role}{'/' + name if name else ''}: {m.content[:200]}")
    return "\n".join(out)

def supervisor(state: SupervisorState) -> Command:
    steps = state.get("steps", 0)
    if steps >= 4:                       
        return Command(goto=END)
    decision = llm.with_structured_output(Route).invoke(
        "You are a supervisor routing a request to the right specialist.\n"
        "- math_expert : does arithmetic and calculations.\n"
        "- ml_tutor    : explains machine-learning concepts in words.\n"
        "Pick who should act NEXT. If a specialist has ALREADY fully answered "
        "the user, pick FINISH.\n\n"
        f"Conversation so far:\n{_render(state['messages'])}")
    nxt = END if decision.next == "FINISH" else decision.next
    print(f"   🧭 Supervisor (step {steps}) → {decision.next}")
    return Command(goto=nxt, update={"steps": steps + 1})

def make_specialist(name: str, system_prompt: str):
    def node(state: SupervisorState) -> Command:
        resp = llm.invoke([SystemMessage(system_prompt)] + state["messages"])
        print(f"   🧑‍🔧 {name} responded")
        return Command(goto="supervisor",      
                       update={"messages": [AIMessage(content=resp.content, name=name)]})
    return node

sg = StateGraph(SupervisorState)
sg.add_node("supervisor", supervisor)
sg.add_node("math_expert", make_specialist(
    "math_expert", "You are a math expert. Compute the exact answer concisely."))
sg.add_node("ml_tutor", make_specialist(
    "ml_tutor", "You are a friendly ML tutor. Explain the concept concisely."))


sg.add_edge(START, "supervisor")
supervisor_app = sg.compile()

print("Q: What is 25 times 4?\n")
res = supervisor_app.invoke(
    {"messages": [HumanMessage("What is 25 times 4?")], "steps": 0})
print("\n🤖 Final answer:", res["messages"][-1].content[:200])

## 🎯 Scenario — When to use a Multi-Agent Supervisor

> **Use it when one task spans *multiple domains* that each need different skills/tools/prompts.**

**Real-world example:** *"A bank's AI assistant gets 'Transfer $500 and explain compound interest.'
The supervisor routes the transfer to a `transactions_agent` (with banking tools + approval gates)
and the explanation to an `education_agent`. One coordinator, two specialists, clean separation —
and you can upgrade either specialist without touching the other."*

**🧪 Try it yourself:** Add a third specialist `code_helper` (writes tiny Python snippets) and ask
*"Explain a for-loop and show me 3 + 5 in Python."* Watch the supervisor route to two specialists.

---
# Pattern 7 — Long-Term Memory (across sessions)

## 🧠 Theory — What & Why

There are **two kinds of memory**, and beginners constantly confuse them:

| | **Short-term** (Checkpointer) | **Long-term** (Store) |
|---|---|---|
| Scope | one conversation (`thread_id`) | the user, **forever** |
| Holds | the message history | **facts** ("name is Priya", "prefers Python") |
| Survives a new chat? | ❌ no | ✅ yes |
| LangGraph object | `MemorySaver` | `InMemoryStore` (→ Redis/DB in prod) |

The previous notebook used the **checkpointer** (short-term). This pattern adds the **Store** —
the agent **remembers facts about you even in a brand-new conversation**, like a real assistant.

### Why it matters
- **Personalisation** — "Welcome back, Priya! Still preparing for the ML exam?"
- **No repetition** — the user doesn't re-explain themselves every session.
- This is how ChatGPT's "Memory" feature and CRM assistants work under the hood.

### How a memory-augmented turn works
```
1. RECALL  → search the Store for facts about this user
2. ANSWER  → give the LLM those facts as context, then respond
3. SAVE    → extract any NEW facts from this turn and write them to the Store
```


In [ ]:
# ── Pattern 7: Long-term memory with a Store ──────────────────────────────────

store = InMemoryStore()                  # in production: Redis / Postgres-backed
USER = ("memories", "user_priya")     # namespace = (category, user_id)

class Fact(BaseModel):
    """A durable fact worth remembering about the user (or none)."""
    has_fact: bool = Field(description="true if the message contains a personal fact to remember")
    fact: str = Field(description="the fact in third person, e.g. 'is preparing for an ML exam'")

extractor = llm.with_structured_output(Fact)

def chat_with_memory(user_text: str) -> str:
    # 1) RECALL — pull everything we know about this user
    known = [m.value["text"] for m in store.search(USER)]
    context = "\n".join(f"- {k}" for k in known) if known else "(nothing yet)"

    # 2) ANSWER — give the LLM the remembered facts as context
    answer = llm.invoke(
        f"Known facts about the user:\n{context}\n\n"
        f"User says: {user_text}\n\n"
        f"Reply warmly and personally, using the facts if relevant.").content

    # 3) SAVE — extract any new fact and store it
    f = extractor.invoke(f"Does this message contain a personal fact to remember? "
                         f"Message: {user_text}")
    if f.has_fact:
        store.put(USER, str(uuid4()), {"text": f.fact})
        print(f"   💾 saved memory: '{f.fact}'")
    return answer

print("──── SESSION 1 (Monday) ────")
print("🤖", chat_with_memory("Hi! I'm Priya and I'm preparing for my ML exam next week.")[:200])
print()
print("🤖", chat_with_memory("I find decision trees really confusing.")[:200])

In [ ]:
# ── A BRAND-NEW session (different day, no chat history) ──────────────────────
# Short-term memory would be empty here. But the STORE persists the facts.

print("Facts currently in long-term memory:")
for m in store.search(USER):
    print(f"   • {m.value['text']}")

print("\n──── SESSION 2 (Friday — fresh conversation) ────")
print("🤖", chat_with_memory("Hey, I'm back! What was I working on?")[:300])

## 🎯 Scenario — When to use Long-Term Memory

> **Use it whenever the agent should *know the user* over time, not just within one chat.**

**Real-world example:** *"A coding assistant remembers that a user already knows Python but struggles
with recursion. Three weeks later, when they ask about trees, it says
'Since trees are recursive — the topic you found tricky — let's use a diagram.' That continuity is
what makes an assistant feel intelligent instead of amnesiac."*

**🧪 Try it yourself:** In Session 2, ask *"Which topic should I review?"* — the agent should
recall that decision trees were confusing and suggest them.

---
# Pattern 8 — Agentic RAG (smart retrieval)

## 🧠 Theory — What & Why

**Classic RAG** *always* retrieves documents, then answers — even for "Hi, how are you?" That wastes
time and can inject irrelevant context. **Agentic RAG** makes retrieval a **decision**:

> *The agent itself decides* — "Do I need to look this up, or do I already know it?" — and after
> retrieving, it can **judge** whether the documents are good enough, and **re-search** if not.

```
                        ┌─────────────────────────────┐
   question ──▶ Agent ──│ Do I need the knowledge base?│
                        └──────┬───────────────┬──────┘
                          yes  │               │ no
                               ▼               ▼
                        retrieve() tool   answer from own knowledge
                               │
                               ▼
                        answer grounded in docs
```

### Why it matters
- **Efficiency** — no pointless retrieval for chit-chat or general questions.
- **Accuracy** — course-specific / private questions get grounded in *your* documents.
- **Self-correcting** — the full version *grades* retrieved docs and rewrites the query if they're weak.

> 💡 This cell shows the **decision** (the heart of agentic RAG) with a tiny in-memory KB so it runs
> instantly. For the **full production version** — real vector store, doc grading, query rewriting,
> CRAG/LangGraph + RAGAS evaluation — see `36_multimodal_rag/production_rag_complete.ipynb`.


In [ ]:
# ── Pattern 8: Agentic RAG — the agent DECIDES whether to retrieve ────────────

# A tiny "knowledge base" = course-specific facts the model can't know on its own
COURSE_KB = {
    "exam date":   "The LogicMojo ML final exam for the Jan26 batch is on March 15, 2026.",
    "instructor":  "The ML course is taught by instructor Suvom at LogicMojo.",
    "passing mark":"A score of 60% is required to pass the LogicMojo ML certification.",
}

@tool
def search_course_kb(query: str) -> str:
    """Search the LogicMojo course knowledge base for facts about THIS specific
    course — exam dates, instructor, grading, schedule. Use this for any question
    about the course itself. Do NOT use it for general ML concepts.
    """
    q = query.lower()
    hits = [v for k, v in COURSE_KB.items() if any(w in q for w in k.split())]
    return "\n".join(hits) if hits else "No course-specific info found."

rag_agent = create_react_agent(
    model=llm,
    tools=[search_course_kb],
    prompt="You are a course assistant. For questions about THIS course (exam, "
           "instructor, grading) you MUST call search_course_kb. For general ML "
           "concepts, answer directly from your own knowledge WITHOUT the tool.",
)

def ask_rag(q):
    r = rag_agent.invoke({"messages": [HumanMessage(q)]})
    retrieved = any(getattr(m, "tool_calls", None) for m in r["messages"])
    print(f"❓ {q}")
    print(f"   {'📚 RETRIEVED from KB' if retrieved else '🧠 answered from own knowledge'}")
    print(f"   🤖 {r['messages'][-1].content[:220]}\n")

# Course-specific → SHOULD retrieve
ask_rag("When is our ML final exam?")
# General concept → should NOT retrieve
ask_rag("In one sentence, what is a neural network?")

## 🎯 Scenario — When to use Agentic RAG

> **Use it when *some* questions need your private documents and *others* don't — and you want the
> agent to tell the difference.**

**Real-world example:** *"A company's internal assistant gets two questions: 'What's our refund
policy?' (must check the private policy docs) and 'What's the capital of France?' (general
knowledge — no retrieval needed). Agentic RAG retrieves only for the first, saving latency and
keeping answers grounded where it counts."*

**🧪 Try it yourself:** Add a fact `"office hours": "..."` to `COURSE_KB` and ask
*"When are office hours?"* vs *"What is gradient descent?"* — confirm only the first retrieves.